# Test EvaluatePhase with Resolved Baseline Patch

In [1]:
import json
import sys
from pathlib import Path

# Run from src directory for correct imports
project_root = Path("/root/makharev/agent-swe-ace")
sys.path.insert(0, str(project_root / "src"))

from phases.evaluate import EvaluatePhase

/root/makharev/agent-swe-ace/.venv/lib/python3.13/site-packages/requests/__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (7.2.0)/charset_normalizer (3.4.6) doesn't match a supported version!
  warnings.warn(


In [14]:
from datasets import load_dataset

dataset = load_dataset("princeton-nlp/SWE-bench_Lite", split="test")
print(f"Loaded {len(dataset)} instances")

instance_id = "astropy__astropy-14995"

instance = None
for item in dataset:
    if item["instance_id"] == instance_id:
        instance = dict(item)
        break

Loaded 300 instances


In [15]:
# Load the patch from baseline results
data_dir = Path("../data/run_baseline_qwen3coder")
results_path = data_dir / "princeton-nlp__SWE-bench_Lite" / "results" / instance_id / "iter_0.json"

with open(results_path) as f:
    result_data = json.load(f)

patch = result_data["patch"]
original_resolved = result_data["resolved"]

print(f"Patch length: {len(patch)} chars")
print(f"Original resolved: {original_resolved}")
print(f"\nPatch preview (first 500 chars):")
print(patch[:500])

Patch length: 118757 chars
Original resolved: True

Patch preview (first 500 chars):
diff --git a/astropy/nddata/mixins/ndarithmetic.py b/astropy/nddata/mixins/ndarithmetic.py
index 4153dfccb..e164a7136 100644
--- a/astropy/nddata/mixins/ndarithmetic.py
+++ b/astropy/nddata/mixins/ndarithmetic.py
@@ -482,6 +482,228 @@ class NDArithmeticMixin:
                 operation, operand, result, correlation, **axis_kwarg
             )
 
+    def _arithmetic_wcs(self, operation, operand, compare_wcs, **kwds):
+        """
+        Calculate the resulting wcs.
+
+        There is actually


In [24]:
# Check if required Docker images exist
import docker
client = docker.from_env()

# Check for instance image
instance_image = f"swebench/sweb.eval.x86_64.astropy_1776_{instance_id.split("__")[1]}:latest"
instance_image_d = client.images.get(instance_image)
print(instance_image_d)

<Image: 'swebench/sweb.eval.x86_64.astropy_1776_astropy-14995:latest'>


In [26]:
from swebench.harness.test_spec.test_spec import make_test_spec
from swebench.harness.docker_build import build_env_images

test_spec = make_test_spec(instance)
build_env_images(client, [test_spec])

env_image = test_spec.env_image_key
env_image_d = client.images.get(env_image)
print(env_image_d)

Building base image (sweb.base.py.x86_64:latest)
Base images built successfully.
Total environment images to build: 1
All environment images built successfully.
<Image: 'sweb.env.py.x86_64.428468730904ff6b4232aa:latest'>


In [27]:
eval_phase = EvaluatePhase(
    use_docker=True,
    timeout=1800,
    rm_image=False,
    output_dir='data',
    run_name="test_eval",
    benchmark="swebench-lite",
)

result = eval_phase.run(
    instance=instance,
    patch=patch,
    iteration=0,
)

# Print results
print("\n" + "=" * 60)
print("Results")
print("=" * 60)
print(f"Instance:         {instance_id}")
print(f"Original resolved: {original_resolved}")
print(f"New resolved:      {result.resolved}")
print(f"Match:            {original_resolved == result.resolved}")
print(f"\nFeedback: {result.feedback[:500]}...")
print(f"\nResult saved to: {result.result_path}")


2026-03-24 01:04:56.062 | INFO     | phases.evaluate:run:81 - [Evaluate] Evaluating patch for astropy__astropy-14995 (iter 0)
2026-03-24 01:04:56.069 | INFO     | evaluation.swebench:validate_patch_docker:51 - Running Docker evaluation for astropy__astropy-14995...
2026-03-24 01:08:09.225 | INFO     | evaluation.swebench:validate_patch_docker:63 - Evaluation result for astropy__astropy-14995: resolved=True
2026-03-24 01:08:09.227 | INFO     | phases.evaluate:run:124 - [Evaluate] astropy__astropy-14995 RESOLVED!
2026-03-24 01:08:09.227 | DEBUG    | data_io.writers:save_result:159 - Saved result to data/swebench-lite/results/astropy__astropy-14995/iter_0.json



Results
Instance:         astropy__astropy-14995
Original resolved: True
New resolved:      True
Match:            True

Feedback: Patch resolved all tests successfully!...

Result saved to: data/swebench-lite/results/astropy__astropy-14995/iter_0.json
